In [3]:
!pip install koreanize_matplotlib

import warnings
import koreanize_matplotlib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import ast

# 경고 무시
warnings.filterwarnings("ignore")
%config lnlineBackend.figure_format = 'retina'

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)  # 출력할 너비를 넉넉하게 조정
pd.set_option('display.expand_frame_repr', False)  # 옆으로 길어져도 줄바꿈 없이 출력
pd.set_option('display.max_colwidth', None)  # 긴 문자열도 생략 없이 출력

try:
	from google.colab import drive
	drive.mount('/content/drive')

	import os
	os.chdir('/content/drive/MyDrive/파트4')
	print('✅ Succesful access google_drive_directory')
	
except Exception as e:
	print('🤗 Hello vscode')

🤗 Hello vscode


In [6]:
API_KEY_PATH ='/home/project_yujin/high_project/sprintda03-yujin.json'

In [70]:
## get_df 함수
API_KEY_PATH ='/home/project_yujin/high_project/sprintda03-yujin.json'

def get_df(db_name, table_name):
    if table_name in ['accounts_user', 'accounts_blockrecord']:
        table_name = pd.read_parquet(
            f"gs://high_project/{db_name}/{table_name}.parquet", 
            storage_options={'token' : API_KEY_PATH})
    else:
        table_name = pd.read_csv(
            f"gs://high_project/{db_name}/{table_name}.csv",
            storage_options={'token' : API_KEY_PATH}
            )
    return table_name
    

## literal_eval 형변환 함수
import ast
def to_literal_eval(df, column):
    return  df[column].apply(lambda x: ast.literal_eval(x) if x != '[]' else [])

    
drop_users = [831956, 1580627, 1580689, 1580626, 995177]
drop_schools = [5948, 5949, 5964]

In [55]:
accounts_user = get_df('votes', 'accounts_user')

In [99]:
## 전처리
# 관리자 제거
accounts_user = accounts_user[~accounts_user['id'].isin(drop_users)]

## new 컬럼
accounts_user['friend_id_list'] = to_literal_eval(accounts_user, 'friend_id_list')
accounts_user['count_friends'] = accounts_user['friend_id_list'].apply(len)

accounts_user['block_user_id_list'] = to_literal_eval(accounts_user, 'block_user_id_list')
accounts_user['count_block_users'] = accounts_user['block_user_id_list'].apply(len)

accounts_user['hide_user_id_list'] = to_literal_eval(accounts_user, 'hide_user_id_list')
accounts_user['count_hide_users'] = accounts_user['hide_user_id_list'].apply(len)

In [100]:
accounts_user = accounts_user.drop(columns=['is_superuser', 'is_staff', 'friend_id_list', 'block_user_id_list', 'hide_user_id_list'])

## 2) accounts_blockrecord

In [101]:
accounts_blockrecord = get_df('votes', 'accounts_blockrecord')

In [102]:
# 컬럼 순서 정리
accounts_blockrecord = accounts_blockrecord[['id', 'user_id', 'block_user_id', 'reason', 'created_at']]

# 시간타입 형변환
accounts_blockrecord['created_at'] = pd.to_datetime(accounts_blockrecord['created_at'])

# ‼️ 중복치 제거
accounts_blockrecord = accounts_blockrecord.loc[:, 'user_id':].drop_duplicates()

# ‼️ 관리자 유저 필터링
accounts_blockrecord = accounts_blockrecord[~accounts_blockrecord['user_id'].isin([831956, 1580627, 1580689, 1580626, 995177])]

# head
accounts_blockrecord.head()

,user_id,block_user_id,reason,created_at
0,878476,867483,그냥...,2023-05-04 23:01:53
1,867564,867190,친구 사이가 어색해짐,2023-05-05 01:17:08
2,875261,875110,나랑 관련 없는 질문을 자꾸 보냄,2023-05-05 01:50:55
3,883511,883696,그냥...,2023-05-05 05:21:52
4,870177,871349,그냥...,2023-05-05 06:40:34


In [73]:
accounts_blockrecord['reason'].unique()

array(['그냥...', '친구 사이가 어색해짐', '나랑 관련 없는 질문을 자꾸 보냄', '기타', '모르는 사람임',
       '너무 많은 양의 질문을 보냄', '사칭 계정'], dtype=object)

In [ ]:
# 유저별 차단 받은수
accounts_blockrecord['block_user_id'].value_counts().reset_index()

,block_user_id,count
0,898020,76
1,1380465,25
2,897681,25
3,877266,25
4,876207,24
...,...,...
16235,1357421,1
16236,844076,1
16237,1222669,1
16238,1370385,1


In [ ]:
accounts_blockrecord
count_blocked_by_no_related_question

,user_id,block_user_id,reason,created_at
0,878476,867483,그냥...,2023-05-04 23:01:53
1,867564,867190,친구 사이가 어색해짐,2023-05-05 01:17:08
2,875261,875110,나랑 관련 없는 질문을 자꾸 보냄,2023-05-05 01:50:55
3,883511,883696,그냥...,2023-05-05 05:21:52
4,870177,871349,그냥...,2023-05-05 06:40:34
...,...,...,...,...
19477,879416,875351,사칭 계정,2024-04-05 08:15:03
19478,1292346,1444256,친구 사이가 어색해짐,2024-04-25 09:28:19
19479,1292346,1379126,친구 사이가 어색해짐,2024-04-25 09:28:24
19480,1583612,1582869,모르는 사람임,2024-05-05 11:06:31


In [122]:
accounts_blockrecord['reason'].unique()

array(['그냥...', '친구 사이가 어색해짐', '나랑 관련 없는 질문을 자꾸 보냄', '기타', '모르는 사람임',
       '너무 많은 양의 질문을 보냄', '사칭 계정'], dtype=object)

In [123]:
accounts_blockrecord['reason'].nunique()

7

In [ ]:
# 총 차단받은수
count_total_blocked = accounts_blockrecord['block_user_id'].value_counts().reset_index()

# 모르는 사람으로 차단받은 수
count_blocked_by_stranger = accounts_blockrecord.query('reason == "모르는 사람임"')['block_user_id'].value_counts().reset_index()

# 사칭 계정으로 차단받은수
count_blocked_by_imposter = accounts_blockrecord.query('reason == "사칭 계정"')['block_user_id'].value_counts().reset_index()

# 너무 많은 질문으로 차단받은수
count_blocked_by_many_question = accounts_blockrecord.query("reason == '너무 많은 양의 질문을 보냄'")['block_user_id'].value_counts().reset_index()

# 질문관련으로 차단받은수
count_blocked_by_no_related_question = accounts_blockrecord.query("reason == '나랑 관련 없는 질문을 자꾸 보냄'")['block_user_id'].value_counts().reset_index()

# 친구 사이가 어색해져서 차단받은수
count_blocked_by_awkward = accounts_blockrecord.query("reason == '친구 사이가 어색해짐'")['block_user_id'].value_counts().reset_index()

# 기타 및 그냥
count_blocked_by_other = accounts_blockrecord.query("reason == '기타' or reason == '그냥'")['block_user_id'].value_counts().reset_index()

# 그냥
count_blocked_by_other = accounts_blockrecord.query("reason == '기타' or reason == '그냥'")['block_user_id'].value_counts().reset_index()

In [140]:
merged_counts = pd.DataFrame()
merged_counts = pd.merge(count_total_blocked, count_blocked_by_stranger, how='outer', on='block_user_id', suffixes=('_total', '_stranger'))
merged_counts = pd.merge(merged_counts, count_blocked_by_imposter, how='outer', on='block_user_id', suffixes=('', '_imposter'))
merged_counts = pd.merge(merged_counts, count_blocked_by_related_question, how='outer', on='block_user_id', suffixes=('', '_related_question'))
merged_counts = pd.merge(merged_counts, count_blocked_by_awkward, how='outer', on='block_user_id', suffixes=('', '_awkward'))
merged_counts = pd.merge(merged_counts, count_blocked_by_other, how='outer', on='block_user_id', suffixes=('', '_other'))

In [114]:
merged_counts.columns

Index(['block_user_id', 'count_total', 'count_stranger', 'count', 'count_related_question', 'count_awkward', 'count_other'], dtype='object')

In [142]:
# 각 행의 count 값들의 합을 계산
merged_counts['calculated_count_total'] = (
    merged_counts[['count_stranger', 'count', 'count_related_question', 'count_awkward', 'count_other']]
    .sum(axis=1, skipna=True)  # NaN을 제외한 합계 계산
)

# 계산된 'calculated_count_total'과 실제 'count_total' 비교
merged_counts['is_total_matching'] = merged_counts['count_total'] == merged_counts['calculated_count_total']

# 결과 출력 (각 행에 대해 합이 맞는지 확인)
merged_counts[['block_user_id', 'count_total', 'calculated_count_total', 'is_total_matching']]


,block_user_id,count_total,calculated_count_total,is_total_matching
0,898020,76,76.0,True
1,1380465,25,25.0,True
2,897681,25,25.0,True
3,877266,25,25.0,True
4,876207,24,24.0,True
...,...,...,...,...
16235,1357421,1,1.0,True
16236,844076,1,1.0,True
16237,1222669,1,1.0,True
16238,1370385,1,1.0,True


In [129]:
merged_counts[merged_counts['is_total_matching'] != True]

,block_user_id,count_total,count_stranger,count,count_related_question,count_awkward,count_other,calculated_count_total,is_total_matching
153,885794,5,NaN,NaN,NaN,NaN,4.0,4.0,False
1632,874270,2,1.0,NaN,NaN,NaN,NaN,1.0,False
4893,867483,1,NaN,NaN,NaN,NaN,NaN,0.0,False
11273,883696,1,NaN,NaN,NaN,NaN,NaN,0.0,False
11274,871349,1,NaN,NaN,NaN,NaN,NaN,0.0,False
11279,887434,1,NaN,NaN,NaN,NaN,NaN,0.0,False


In [137]:
accounts_blockrecord[accounts_blockrecord['block_user_id'].isin([867483])]

,user_id,block_user_id,reason,created_at
0,878476,867483,그냥...,2023-05-04 23:01:53


In [139]:
accounts_blockrecord[accounts_blockrecord['block_user_id'].isin([885794])]

,user_id,block_user_id,reason,created_at
8,879662,885794,기타,2023-05-05 13:04:31
9,879662,885794,기타,2023-05-05 13:04:42
10,879662,885794,그냥...,2023-05-05 13:04:52
11,879662,885794,기타,2023-05-05 13:04:56
12,879662,885794,기타,2023-05-05 13:05:01
